# MRR

Compare Monthly Recurring Revenue computed by the Tidemill engine with
ground-truth values derived directly from Stripe subscription objects.

## How Stripe stores recurring prices

Every Stripe **Subscription** contains one or more **SubscriptionItems**,
each referencing a **Price** object.  The Price defines:

| Field                       | Meaning                                             |
|-----------------------------|-----------------------------------------------------|
| `unit_amount`               | Price in smallest currency unit (e.g. 2100 = $21)   |
| `recurring.interval`        | Billing cycle: `day`, `week`, `month`, or `year`    |
| `recurring.interval_count`  | Intervals per cycle (e.g. 3 → quarterly if monthly) |

A SubscriptionItem also carries a `quantity` (default 1), so the total
charge per cycle is `unit_amount × quantity`.

## How MRR is normalised

MRR converts every billing interval to a **monthly** rate:

| Interval | Formula                              |
|----------|--------------------------------------|
| month    | `amount / interval_count`            |
| year     | `amount / (12 × interval_count)`     |
| week     | `amount × 52 / (12 × interval_count)`|
| day      | `amount × 365 / (12 × interval_count)`|

Only subscriptions with `status == "active"` contribute to current MRR.
All arithmetic uses integer cents to avoid floating-point drift.

In [41]:
import os

import stripe

from tidemill import reports
from tidemill.reports.stripecheck import StripeData, TidemillClient
from tidemill.reports.stripecheck import compare as cmp

stripe.api_key = os.environ["STRIPE_API_KEY"]
reports.setup()

START, END = "2025-06-01", "2026-03-31"
tm = TidemillClient()
sd = StripeData()
print(f"Stripe subscriptions: {sd.summary()}")

Stripe subscriptions: 38 total, 24 active, 13 canceled, 1 incomplete_expired


## 1. MRR comparison — Tidemill vs Stripe

`stripecheck.compare.mrr()` fetches Tidemill's MRR via the REST API
and independently sums `mrr_cents` over active Stripe subscriptions.
Any difference indicates a divergence in event processing.

In [42]:
reports.mrr.style_stripe_comparison(reports.mrr.stripe_comparison(tm, sd, at="2026-03-01"))

MRR (Tidemill),MRR (Stripe),Difference,Match,ARR
$967.33,$967.33,$0.00,True,"$11,607.96"


## 2. Per-subscription MRR breakdown

Drill into every Stripe subscription to see its individual MRR
contribution.  `stripecheck.stripe_metrics.subscription_mrr()` reads
`items.data[*].price` on each subscription and applies the normalisation
table from above.  Useful for identifying *which* subscription diverges
when the totals don't match.

In [43]:
cmp.per_subscription_mrr(sd)

,id,customer,status,mrr_cents,currency,mrr
0,sub_1TMpsiCMLOTbZAd7RSlLDpaN,cus_ULWwTgCTM36v4e,active,20750,usd,$207.50
1,sub_1TMpssCMLOTbZAd7sJVjofN3,cus_ULWwO9uqpSPVUt,active,7900,usd,$79.00
2,sub_1TMpt7CMLOTbZAd70vkGqRdG,cus_ULWwQexwEAjgi3,active,7900,usd,$79.00
3,sub_1TMpsTCMLOTbZAd7HBxFZ83M,cus_ULWwNaOa2hh32z,active,7900,usd,$79.00
4,sub_1TMpsYCMLOTbZAd7tetMCehL,cus_ULWwaKJiAP4F85,active,7900,usd,$79.00
5,sub_1TMpt2CMLOTbZAd7CIeWH9xX,cus_ULWwScgoS7pqiF,canceled,7900,usd,$79.00
6,sub_1TMpsdCMLOTbZAd7RDgGSf9l,cus_ULWwaIAUzeOUbH,active,6583,usd,$65.83
7,sub_1TMptUCMLOTbZAd7tVFsbBif,cus_ULWxn3RhYPwGHW,active,2100,usd,$21.00
8,sub_1TMptRCMLOTbZAd7w0HxbiKG,cus_ULWxvW300Pmlkv,active,2100,usd,$21.00
9,sub_1TMptMCMLOTbZAd7yzOlZf4p,cus_ULWx9JMHHI7ZBv,incomplete_expired,2100,usd,$21.00


## 3. MRR Breakdown (movements)

Tidemill classifies every MRR change into a **movement type**:

| Movement      | Meaning                                              |
|---------------|------------------------------------------------------|
| **new**       | First subscription for a customer                    |
| **expansion** | Upgrade or additional subscription                   |
| **contraction** | Downgrade (lower plan/fewer seats)                 |
| **churn**     | Cancellation — customer lost entirely                |
| **reactivation** | Previously churned customer returns               |

The breakdown endpoint sums these over the full period.

In [44]:
reports.mrr.plot_breakdown(reports.mrr.breakdown(tm, START, END))

## 4. Monthly MRR with movement split

Query MRR breakdown per month to show how new, expansion, contraction, and churn compose each month's MRR change.

In [45]:
wf = reports.mrr.waterfall(tm, START, END)
reports.mrr.style_waterfall(wf)

,starting_mrr,new,expansion,reactivation,contraction,churn,net_change,ending_mrr
month,,,,,,,,
2025-06,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00
2025-07,$0.00,$878.33,$0.00,$0.00,$0.00,$0.00,$878.33,$878.33
2025-08,$878.33,$84.00,$58.00,$0.00,$0.00,$-21.00,$121.00,$999.33
2025-09,$999.33,$0.00,$0.00,$0.00,$-58.00,$-63.00,$-121.00,$878.33
2025-10,$878.33,$84.00,$0.00,$0.00,$0.00,$-21.00,$63.00,$941.33
2025-11,$941.33,$0.00,$58.00,$0.00,$-58.00,$-100.00,$-100.00,$841.33
2025-12,$841.33,$168.00,$0.00,$0.00,$0.00,$-42.00,$126.00,$967.33
2026-01,$967.33,$84.00,$0.00,$0.00,$0.00,$-42.00,$42.00,"$1,009.33"
2026-02,"$1,009.33",$0.00,$0.00,$0.00,$0.00,$-42.00,$-42.00,$967.33


### 4a. Per-customer movement log

Drill into the individual movements that compose each month's waterfall.
Every row is a single customer's MRR change — **new**, **expansion**,
**reactivation**, **contraction**, or **churn** — with monthly subtotals
that should match the waterfall table above.

In [46]:
ml = reports.mrr.movement_log(tm, START, END)
reports.mrr.style_movement_log(ml)

month,date,customer,customer_id,movement,amount
2025-07,2025-07-01,Active Annual Enterprise #7,cus_ULWwTgCTM36v4e,new,$207.50
2025-07,2025-07-01,Active Annual Pro #6,cus_ULWwaIAUzeOUbH,new,$65.83
2025-07,2025-07-01,Active Monthly Pro #4,cus_ULWwNaOa2hh32z,new,$79.00
2025-07,2025-07-01,Active Monthly Pro #5,cus_ULWwaKJiAP4F85,new,$79.00
2025-07,2025-07-01,Active Starter #1,cus_ULWwrTRn9Nc2KQ,new,$21.00
2025-07,2025-07-01,Active Starter #2,cus_ULWwH5W4Rtwl58,new,$21.00
2025-07,2025-07-01,Active Starter Heavy #3,cus_ULWwJAvHyIN6uT,new,$21.00
2025-07,2025-07-01,Churned Pro #11,cus_ULWwScgoS7pqiF,new,$79.00
2025-07,2025-07-01,Churned Starter #8,cus_ULWwpTRPX25YZq,new,$21.00
2025-07,2025-07-01,Downgraded Pro→Starter #10,cus_ULWwfk0VwelKjT,new,$79.00


In [47]:
reports.mrr.plot_waterfall(wf)

In [48]:
reports.mrr.plot_trend(reports.mrr.trend(tm, START, END))

## 5. MRR by subscription status (Stripe)

Stripe subscriptions move through lifecycle states:

| Status               | Pays? | Notes                                       |
|----------------------|-------|---------------------------------------------|
| `trialing`           | No    | Free trial period, no charge yet            |
| `active`             | Yes   | Current, paying — contributes to MRR        |
| `past_due`           | Retry | Payment failed, Stripe retrying             |
| `canceled`           | No    | Terminated — `canceled_at` is set           |
| `unpaid`             | No    | Retries exhausted, still open               |
| `incomplete`         | No    | First payment pending                       |
| `incomplete_expired` | No    | First payment never completed               |

Only `active` subscriptions are counted toward MRR.

In [49]:
status_df = reports.mrr.stripe_status_breakdown(sd)
reports.mrr.style_stripe_status_breakdown(status_df)

status,count,mrr
active,24,$967.33
canceled,13,$331.00
incomplete_expired,1,$21.00


In [50]:
reports.mrr.plot_stripe_status_breakdown(status_df)